In [ ]:
%pip install semantic-link-labs

In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
import notebookutils 
import sempy.fabric as fabric
import sempy_labs as labs
import sempy_labs.admin as admin
import sempy_labs.graph as graph
import sempy_labs.notebook as sempy_notebooks
import sempy_labs.variable_library as var_lib
import pandas as pd
import numpy as np
import requests
import os

In [ ]:
workspace_name = fabric.resolve_workspace_name()

# Instructions

Import this notebook to your Fabric workspace, [run it](#automatic-bits), perform the [manual](#manual-bits) steps, put some data into storage containers, and you'll be ready to go!!

For data shape, use the API / SLL response to the relevant Get APIs. Refer to the ReadMe and function descriptions of each notebook for more information 

# Automatic bits

## INSTRUCTIONS: Placeholder variables

**READ THIS**

In the below code you will find placeholder values indicated by `[...]`, you will need to replace these (including the `[` `]` symbols) with values from your own tenant.

### Managed Private Endpoints

These allow your workspace to communicate out to Private Network protected resources. You will need to establish these in all your Workspaces _(dev, uat, stg, prd, ...)_


The below code will request that resources be created for the current workspace, and the Azure Resource owner will need to approve the MPE's under the resource's networking menu. 

For more info see [Microsoft's documentation](https://learn.microsoft.com/en-au/fabric/security/security-managed-private-endpoints-create)

In [ ]:
## REPLACE THE PLACEHOLDER VALUES <...>

# create the storage container MPE
labs.managed_private_endpoint.create_managed_private_endpoint(
    (workspace_name + "-adlg2"), 
    [ADLG2 container resource ID], # TODO:replace this
    "dfs",
    ("Managed Private Endpoint for Fabric workspace: " + workspace_name)
)

# create the keyvault MPE
labs.managed_private_endpoint.create_managed_private_endpoint(
    (workspace_name + "-kv"), 
    [Key Vault resource ID], # TODO: replace this
    "vault",
    ("Managed Private Endpoint for Fabric workspace: " + workspace_name)
)

### This section goes to the source GitHub and grabs all the relevant notebook files

There's an anonymous version of the get files function, but if you hit rate-limits, you will need to replace the `[GITHUB PAT HERE]` with a [Personal Access Token for your GitHub account](https://github.com/settings/personal-access-tokens).


There are 3 layers in the notebook hierarchy

Layer | Purpose
:----:|---------
00-environment | **Configuration files, environment variables, ...** <br>All the things you will need to load to get started, regardless of what you're working on
01-utils | **Abstracted functions and processes**<br>Heavy lifting is done here so other notebooks can reference them with simple interfaces, i.e. mounting Azure Data Lake storage, defining API queries & authentication to get Workspace access, ... 
02-instances | **Invoked actions**<br>Pipelines, Executed notebooks (which reference upstream), scheduled items, ...

In [ ]:
# Replace with your GitHub username and repository name
owner = "Argel-Tal"
repo = "fabric-configuration-management"
branch = "main"
# Optional: Specify a path within the repository
path = "" # Empty string for the root directory

raw_file_path = f"https://raw.githubusercontent.com/{owner}/{repo}/refs/heads/{branch}/"

search_string = "notebooks"
file_extension = '.ipynb'

# Replace with your actual Personal Access Token (PAT)- don't save this token to Git!!! Replace with dummy values if you save this notebook
GITHUB_TOKEN = '[GITHUB PAT HERE]'


In [ ]:
# version of get files from github which is PAT-less
def list_files_in_repo_anon(owner, repo, path):
    """Lists all files in a public GitHub repository using the API."""
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
    headers = {
        "Accept": "application/vnd.github.v3+json"
    }

    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        contents = response.json()
        files = []
        for item in contents:
            # Recursively handle directories to get all files
            if item['type'] == 'dir':
                files.extend(list_files_in_repo_anon(owner, repo, item['path']))
            else:
                files.append(item['path'])
        return files
    else:
        print(f"Failed to get contents: {response.status_code} - {response.text}")
        return []

# version of the function which uses a PAT token to read files! Less rate limited
def list_files_in_repo(owner, repo, path, token):
    """Lists all files in a public GitHub repository using the API."""
    url = f"https://api.github.com/repos/{owner}/{repo}/contents/{path}"
    headers = {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json"
    }

    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        contents = response.json()
        files = []
        for item in contents:
            # Recursively handle directories to get all files
            if item['type'] == 'dir':
                files.extend(list_files_in_repo(owner, repo, item['path'], token))
            else:
                files.append(item['path'])
        return files
    else:
        print(f"Failed to get contents: {response.status_code} - {response.text}")
        return []


In [ ]:
'''
If for some reason you are getting hit wiht GitHub rate limits for # of anonymous API queries:
1. put your PAT in the block above
2. Comment out the line with    `list_files_in_repo_anon()`
3. Uncomment the line with      `list_files_in_repo()`
'''

file_list = list_files_in_repo_anon(owner, repo, path)
# file_list = list_files_in_repo(owner, repo, path, GITHUB_TOKEN)
files_df = pd.DataFrame(file_list, columns=['filename'])
# santitise any spaces in filenames
files_df['filename'] = files_df['filename'].str.replace(" ", "-")

In [ ]:
# process the dataframe so it's file and path, so we can create notebooks in specific directories
fabric_config_notebooks = files_df[files_df["filename"].str.find(search_string) != -1]
fabric_config_notebooks = pd.DataFrame(fabric_config_notebooks['filename'].str.rsplit('/', n = 1, expand = True))
fabric_config_notebooks.columns = ["path", "filename"]
fabric_config_notebooks= fabric_config_notebooks.reset_index(drop=True)
display(fabric_config_notebooks)

In [ ]:
# this installs all the notebooks
for index, row in fabric_config_notebooks.iterrows():
    address = f"{raw_file_path}/{row['path']}/{row['filename']}"
    sempy_notebooks.import_notebook_from_web(
        notebook_name = row['filename'].rsplit(file_extension)[0],
        url = address,
        folder = row['path'].split('notebooks/')[1]
    )

# Manual bits

## Environment

These notebooks rely on the [Semantic Link Labs package](https://semantic-link-labs.readthedocs.io/en/stable/index.html) ([Github](https://github.com/microsoft/semantic-link-labs/tree/main)). This notebook uses `%pip install` to load it, but when running in Pipelines and such you will need to make a Workspace Environment. Currently SLL does not support adding packages to an Environment (the API does). 

Environments are created as workspace artifacts, and then attached to each notebook using metadata, or the bar along the top in the notebook editor.

**Required libraries:**

Library | Version
--------|--------
Fabric  | 3.2.2
semantic-link-labs | 0.13.2

## Variables

In this module, there is an assumed Azure Data Lake Gen2 enabled Azure Storage Account. 

Within this Account, it assumes a Container holding configuration data, and also another container with a subset of workspaces & domains used for testing. This enables you to run these modules on a small list of workspaces (files in the dev subdirectory), refine these modules, add to them, ... while running them in a limited blast-radius mode. Then when you're ready, you can push your content to a Production workspace, update the active variable set using a Fabric Deployment Pipeline, and all runs in the Production Workspace will automatically reference the longer list, and affect those Workspaces.

I used the names: 
- Production: `fabric-management`
- Testing: `dev-fabric-management`

Other useful variables are also stored in a variable library as follows:


name | description | value type | variable value  | Alternate value _(`Dev` workspace, if blank use default)_
:---:|-------------|------------|-----------------|-------
tenant_id | ID of your tenancy | GUID | [YOUR TENANT ID GOES HERE] |
common_capacity | ID of main Fabric capacity (assumes there's a default you assign to if assigning) | GUID | [YOUR PRIMARY FABRIC CAPACITY ID GOES HERE] |
fabric_admins | security group containing fabric admins | String | [YOUR FABRIC ADMINS GROUP ID GOES HERE] |
admin_storage_account | ADLG2 Storage Account name, which contains the configuration files | String | [YOUR ADMIN STORAGE ACCOUNT NAME GOES HERE] | 
admin_filepath | path inside the container where configuration files are stored | String | `fabric-management` | `dev-fabric-management`
admin_sp_app_id | App Reg ID of Service principal which can call Fabric Admin APIs" | GUID | [YOUR ADMIN SERVICE PRINCIPAL APP ID GOES HERE] |
keyvault_uri | URI of the Key Vault | String | `https://[YOUR KEY VAULT NAME].vault.azure.net/` | 
orchestration_sp_app_id | App Reg ID of Service principal which has access to data and is used to run workspace content | GUID | [YOUR ORCHESTRATION SERVICE PRINCIPAL APP ID GOES HERE]
orchestration_sp_app_object_id | Object ID of Service principal which has access to data and is used to run workspace content - needed for assignments | GUID | [YOUR ORCHESTRATION SERVICE PRINCIPAL OBJECT ID GOES HERE]
